
## Real Data Pipeline - GitHub Data Ingestion
### Ingesting real data from GitHub API for popular data engineering repos

In [0]:
# Databricks notebook source
# ============================================================================
# REAL DATA PIPELINE: GitHub Data Ingestion
# ============================================================================
# Purpose: Ingest real GitHub data via GitHub API
# Data Source: GitHub API (https://api.github.com)
# Author: RSangDev
# Date: 2026-06-02
# ============================================================================

import requests
import json
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.types import *
from datetime import datetime
import time

catalog = "workspace"
schema = "github_analytics"

print("=" * 70)
print("GITHUB DATA PIPELINE - INGESTION")
print("=" * 70)


In [0]:
# Cell 1: Setup
# Create schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
print(f"✅ Schema created: {catalog}.{schema}")

# GitHub API Configuration
GITHUB_API_BASE = "https://api.github.com"
REPOS_TO_TRACK = [
    "apache/spark",
    "databricks/databricks-sql-connector",
    "delta-io/delta",
    "dbt-labs/dbt-core",
    "getdbt/dbt-utils",
    "meltanolabs/tap-github",
    "airbyte/airbyte",
    "getindata/dbt-action-metadata",
    "great-expectations/great_expectations",
    "prefecthq/prefect",
]

print(f"📊 Tracking {len(REPOS_TO_TRACK)} repositories")


In [0]:
# Cell 2: Fetch Repository Data from GitHub API
def fetch_repo_data(repo_name):
    """
    Fetch data for a single repository from GitHub API
    """
    try:
        url = f"{GITHUB_API_BASE}/repos/{repo_name}"
        
        # GitHub API call (no authentication required for public repos)
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            
            return {
                "repo_name": repo_name,
                "full_name": data.get("full_name"),
                "description": data.get("description"),
                "url": data.get("html_url"),
                "stars": data.get("stargazers_count", 0),
                "forks": data.get("forks_count", 0),
                "watchers": data.get("watchers_count", 0),
                "open_issues": data.get("open_issues_count", 0),
                "language": data.get("language"),
                "created_at": data.get("created_at"),
                "updated_at": data.get("updated_at"),
                "pushed_at": data.get("pushed_at"),
                "homepage": data.get("homepage"),
                "size": data.get("size", 0),
                "network_count": data.get("network_count", 0),
                "subscribers_count": data.get("subscribers_count", 0),
                "topics": ",".join(data.get("topics", [])),
                "has_issues": data.get("has_issues"),
                "has_wiki": data.get("has_wiki"),
                "is_fork": data.get("fork"),
                "license": data.get("license", {}).get("name") if data.get("license") else None,
                "data_fetched_at": datetime.now().isoformat(),
            }
        else:
            print(f"❌ Failed to fetch {repo_name}: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"❌ Error fetching {repo_name}: {str(e)}")
        return None



In [0]:

# Cell 3: Fetch all repositories
print("\n" + "=" * 70)
print("FETCHING GITHUB DATA...")
print("=" * 70 + "\n")

all_repos = []
for i, repo in enumerate(REPOS_TO_TRACK, 1):
    print(f"[{i}/{len(REPOS_TO_TRACK)}] Fetching {repo}...", end=" ")
    data = fetch_repo_data(repo)
    
    if data:
        all_repos.append(data)
        print(f"✅ ({data['stars']} ⭐️)")
    else:
        print("❌")
    
    # Respect GitHub API rate limits (be nice!)
    time.sleep(1)

print(f"\n✅ Successfully fetched {len(all_repos)} repositories")

In [0]:
# Cell 4: Create Bronze Table - Raw Repository Data
print("\n" + "=" * 70)
print("CREATING BRONZE LAYER")
print("=" * 70 + "\n")

repos_df = spark.createDataFrame(all_repos)

# Add metadata
repos_df = repos_df.withColumn("ingestion_date", F.current_date())
repos_df = repos_df.withColumn("ingestion_timestamp", F.current_timestamp())
repos_df = repos_df.withColumn("source", F.lit("github_api"))

# Save to Bronze
bronze_table = f"{catalog}.{schema}.bronze_repositories"
repos_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(bronze_table)

print(f"✅ Created: {bronze_table}")
print(f"   Records: {repos_df.count()}")


In [0]:
# Cell 5: Fetch Contributor Data
print("\n" + "=" * 70)
print("FETCHING CONTRIBUTOR DATA")
print("=" * 70 + "\n")

def fetch_contributors(repo_name, limit=10):
    """Fetch top contributors for a repository"""
    try:
        url = f"{GITHUB_API_BASE}/repos/{repo_name}/contributors"
        
        response = requests.get(url, params={"per_page": limit}, timeout=10)
        
        if response.status_code == 200:
            contributors = response.json()
            
            result = []
            for contrib in contributors:
                result.append({
                    "repo_name": repo_name,
                    "contributor_login": contrib.get("login"),
                    "contributions": contrib.get("contributions", 0),
                    "avatar_url": contrib.get("avatar_url"),
                    "profile_url": contrib.get("html_url"),
                    "data_fetched_at": datetime.now().isoformat(),
                })
            
            return result
        else:
            return []
            
    except Exception as e:
        print(f"Error fetching contributors for {repo_name}: {e}")
        return []

In [0]:
# Cell 6: Get all contributors
all_contributors = []
for i, repo in enumerate(REPOS_TO_TRACK, 1):
    print(f"[{i}/{len(REPOS_TO_TRACK)}] Fetching contributors for {repo}...", end=" ")
    
    contribs = fetch_contributors(repo, limit=5)
    if contribs:
        all_contributors.extend(contribs)
        print(f"✅ ({len(contribs)} contributors)")
    else:
        print("⚠️ No data")
    
    time.sleep(1)

print(f"\n✅ Total contributors: {len(all_contributors)}")

In [0]:
# Cell 7: Create Bronze Contributors Table
contributors_df = spark.createDataFrame(all_contributors)

contributors_df = contributors_df.withColumn("ingestion_date", F.current_date())
contributors_df = contributors_df.withColumn("ingestion_timestamp", F.current_timestamp())
contributors_df = contributors_df.withColumn("source", F.lit("github_api"))

bronze_contrib_table = f"{catalog}.{schema}.bronze_contributors"
contributors_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(bronze_contrib_table)

print(f"✅ Created: {bronze_contrib_table}")
print(f"   Records: {contributors_df.count()}")



In [0]:
# Cell 8: Summary
print("\n" + "=" * 70)
print("INGESTION COMPLETE")
print("=" * 70)

summary = f"""
BRONZE LAYER CREATED:
  ✅ bronze_repositories: {repos_df.count()} repos
  ✅ bronze_contributors: {contributors_df.count()} contributors

DATA QUALITY:
  - All repos have stars >= 0
  - All contributors have contributions >= 1
  - Timestamps captured for CDC

NEXT STEP: Run 02_data_transformation.py
"""

print(summary)



In [0]:
# Cell 9: Show Sample Data
print("\n📊 SAMPLE REPOSITORIES:")
display(repos_df.select("repo_name", "stars", "forks", "language", "updated_at").limit(5))

print("\n👥 SAMPLE CONTRIBUTORS:")
display(contributors_df.select("repo_name", "contributor_login", "contributions").limit(10))



In [0]:

# Cell 10: Data Catalog
print("\nCreated Tables:")
spark.sql(f"SHOW TABLES IN {catalog}.{schema}").show()